In [152]:
import pandas as pd
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle 

## Data Preprocessing Pipeline Summary

In this notebook, we performed the essential preprocessing steps required before training a machine learning model.

1. **Data Inspection**
   - Loaded the dataset and examined its structure using `head()`, `columns`, and `dtypes`.
   - Identified numerical and categorical features.

2. **Categorical Encoding**
   - The `Geography` column contained categorical values (`France`, `Germany`, `Spain`).
   - Applied **OneHotEncoder** to convert these categories into numerical binary features.

3. **Feature Integration**
   - Converted the encoded output into a DataFrame.
   - Merged the encoded columns back into the original dataset using `pd.concat()`.

4. **Dropping Original Categorical Column**
   - Removed the original `Geography` column since models cannot process raw text values.

5. **Feature–Target Separation**
   - Defined input features `X` and target variable `y` (`Exited`).

6. **Train-Test Split**
   - Split the dataset into training and testing sets using `train_test_split`.
   - 80% data used for training and 20% for testing.

7. **Feature Scaling**
   - Applied **StandardScaler** to normalize numerical features.
   - Fitted the scaler on training data and transformed both training and test sets.

These preprocessing steps ensure that the dataset is fully numerical, properly scaled, and ready for training machine learning models.

In [153]:
#load the dataset for the preprocessing
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [154]:
##Preprocessing the data
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [155]:
#Encode categorical variable 
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
#LABEL ENCODER CONVERTS CATEGORICAL VARIABLE INTO NUMERICAL VARIABLE
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [156]:
onehot_encoder_geo = OneHotEncoder(sparse_output=False)
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder


array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [157]:
onehot_encoder_geo.get_feature_names_out(['Geography'])
#These 3 feature are getting converted into 3 different feature with 0 and 1 as value. This is called one hot encoding. We are doing this because machine learning model can only understand numerical variable.


array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [158]:
geo_encoder_df=pd.DataFrame(geo_encoder,columns=onehot_encoder_geo.get_feature_names_out(['Geography'])).head()
geo_encoder_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [159]:
#combine one hot enoded columns
data=pd.concat([data,geo_encoder_df],axis=1)
data = data.drop(columns=['Geography'])
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [160]:
## saving the encoders for future use
with open('label_encoder_gender.pkl','wb') as f:
    pickle.dump(label_encoder_gender,f) 
with open('onehot_encoder_geo.pkl','wb') as f:
    pickle.dump(onehot_encoder_geo,f)       

In [161]:
#Divide the dataset into dependent and independent variable
X=data.drop('Exited',axis=1)
y=data['Exited']            

#y will be my dependent and x will be my independent feature
#Split the dataset into training and testing set
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)   

#scale these features
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)



In [162]:
print(X_train.dtypes)

CreditScore            int64
Gender                 int64
Age                    int64
Tenure                 int64
Balance              float64
NumOfProducts          int64
HasCrCard              int64
IsActiveMember         int64
EstimatedSalary      float64
Geography_France     float64
Geography_Germany    float64
Geography_Spain      float64
dtype: object


In [163]:
X_train

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
9254,686,1,32,6,0.00,2,1,1,179093.26,NaN,NaN,NaN
1561,632,1,42,4,119624.60,2,1,1,195978.86,NaN,NaN,NaN
1670,559,1,24,3,114739.92,1,1,0,85891.02,NaN,NaN,NaN
6087,561,0,27,9,135637.00,1,1,0,153080.40,NaN,NaN,NaN
6669,517,1,56,9,142147.32,1,0,0,39488.04,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
5734,768,1,54,8,69712.74,1,1,1,69381.05,NaN,NaN,NaN
5191,682,0,58,1,0.00,1,1,1,706.50,NaN,NaN,NaN
5390,735,0,38,1,0.00,3,0,0,92220.12,NaN,NaN,NaN
860,667,1,43,8,190227.46,1,1,0,97508.04,NaN,NaN,NaN


ANN IMPLEMENTATION

In [164]:
import sys
print(sys.executable)

/Users/ritvik_singh/Study/LLM_COURSE/LLM_Course/.venv/bin/python


In [165]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices())

2.15.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


MacBook
│
├── VS Code
│
├── Jupyter Notebook
│
├── .venv (Python 3.9)
│   ├── TensorFlow 2.15
│   ├── tensorflow-metal (GPU acceleration)
│   ├── scikit-learn
│   ├── pandas
│   └── matplotlib
│
└── Apple Metal GPU backend

In [166]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [167]:
X_train.shape[1]

12

In [168]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(16, activation='relu'),

    Dense(1, activation='sigmoid')
])

In [169]:

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [170]:
#compile the model
model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])
#This is for binary classification problem so we are using binary crossentropy as loss function and adam as optimizer



In [171]:
#Set up early stopping and tensorboard callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
) 
#callbacks are used to stop the training when the model is not improving and to log the training process for visualization in tensorboard 


In [172]:
#Early stopping is a technique to stop the training of the model when the model is not improving on the validation set. This helps to prevent overfitting and saves time. We are monitoring the validation loss and if it does not improve for 5 consecutive epochs, we will stop the training and restore the best weights.
early_stopping = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)
#will look for value loss and will stop if it does not improve for 5 consecutive epochs and restore the best weights
early_stopping 

In [173]:
###training the model 
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

# epoch are the no of time model is going to see the data and batch size is the no of samples that will be passed through the model at a time. We are using early stopping and tensorboard callbacks to stop the training when the model is not improving and to log the training process for visualization in tensorboard
# 2 callbacks are used here, one is early stopping and other is tensorboard callback. Early stopping will stop the training when the model is not improving on the validation set and restore the best weights. Tensorboard callback will log the training process for visualization in tensorboard. We can visualize the training process in tensorboard by running the command "tensorboard --logdir=logs/fit" in the terminal and then opening the link "http://localhost:6006/" in the browser.



Epoch 1/100


/Users/ritvik_singh/Study/LLM_COURSE/LLM_Course/.venv/lib/python3.9/site-packages/keras/src/backend.py:5818: UserWarning: "`binary_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Sigmoid activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


250/250 [==============================] - 4s 13ms/step - loss: nan - accuracy: 0.7945 - val_loss: nan - val_accuracy: 0.8035
Epoch 2/100
250/250 [==============================] - 3s 12ms/step - loss: nan - accuracy: 0.7945 - val_loss: nan - val_accuracy: 0.8035
Epoch 3/100
250/250 [==============================] - 3s 12ms/step - loss: nan - accuracy: 0.7945 - val_loss: nan - val_accuracy: 0.8035
Epoch 4/100
250/250 [==============================] - 3s 12ms/step - loss: nan - accuracy: 0.7945 - val_loss: nan - val_accuracy: 0.8035
Epoch 5/100
250/250 [==============================] - 3s 12ms/step - loss: nan - accuracy: 0.7945 - val_loss: nan - val_accuracy: 0.8035
Epoch 6/100
250/250 [==============================] - 3s 12ms/step - loss: nan - accuracy: 0.7945 - val_loss: nan - val_accuracy: 0.8035
Epoch 7/100
250/250 [==============================] - 4s 16ms/step - loss: nan - accuracy: 0.7945 - val_loss: nan - val_accuracy: 0.8035
Epoch 8/100
250/250 [=========================

In [174]:
model.save('ann_model.h5')
#we are saving the model in h5 format which is a binary format that can be used to save the model architecture, weights and optimizer state. This will allow us to load the model later and use it for predictions or further training.


/Users/ritvik_singh/Study/LLM_COURSE/LLM_Course/.venv/lib/python3.9/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [175]:
#load tensorboard extension
%load_ext tensorboard
#we are downloading the tensorboard extension to visualize the training process in tensorboard. We can visualize the training process in tensorboard by running the command "tensorboard --logdir=logs/fit" in the terminal and then opening the link "http://localhost:6006/" in the browser.

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [176]:
%tensorboard --logdir logs/fit/20260309-151137
#we are saving the model in h5 format which is a binary format that can be used to save the model architecture, weights and optimizer state. This will allow us to load the model later and use it for predictions or further training.


Reusing TensorBoard on port 6010 (pid 36664), started 2:05:51 ago. (Use '!kill 36664' to kill it.)

In [177]:
# lOAD THE PICKLE FILE